
## Statistical Analysis of Sales Data

 Research Questions:

 1. Does discount significantly affect sales?
 2. Does profit differ significantly across markets?
 3. Does sales differ significantly across regions?
 4. Does shipping mode affect shipping cost?
 5. Is there a significant relationship between quantity and profit?
 6. Does profit differ significantly across product categories?

In [91]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [92]:
fact = pd.read_excel("Data/FactSales.xlsx")
product = pd.read_excel("Data/DimProduct.xlsx")
geography = pd.read_excel("Data/DimGeography.xlsx")

print("FactSales:", fact.shape)
print("DimProduct:", product.shape)
print("DimGeography:", geography.shape)

FactSales: (49670, 20)
DimProduct: (10246, 4)
DimGeography: (25033, 5)


In [93]:
df = fact.copy()

df = df.merge(
    product[["Product ID", "Category"]],
    on="Product ID",
    how="left"
)

df = df.merge(
    geography[["Order ID", "Region"]],
    on="Order ID",
    how="left"
)

df.head()

,Row ID,Order ID,Product ID,Sales,Quantity,Discount,Profit,Shipping Cost,Customer ID,Order Priority,...,Returned,Shipping ID,Ship Date,Ship Mode,Total Sales Raw,Total Sales Final,Discounted,Profitable,Category,Region
0,3,MX-2012-155047,FUR-BO-10002352,193.279999,2,0.0,54.080002,9.627,KW-16570,Medium,...,0,34096,"Saturday, October 20, 2012",Standard Class,386.559998,386.559998,0,1,Furniture,South
1,5,MX-2012-155047,OFF-AR-10004594,71.599998,2,0.0,11.440000,3.787,KW-16570,Medium,...,0,34096,"Saturday, October 20, 2012",Standard Class,143.199997,143.199997,0,1,Office Supplies,South
2,6,MX-2012-155047,OFF-EN-10001375,56.119999,2,0.0,21.320000,4.718,KW-16570,Medium,...,0,34096,"Saturday, October 20, 2012",Standard Class,112.239998,112.239998,0,1,Office Supplies,South
3,7,MX-2013-134096,OFF-EN-10001375,56.119999,2,0.0,21.320000,4.108,DP-13000,Medium,...,0,30127,"Tuesday, October 1, 2013",Standard Class,112.239998,112.239998,0,1,Office Supplies,South
4,10,MX-2013-134096,TEC-AC-10001830,341.519989,2,0.0,13.640000,17.341,DP-13000,Medium,...,0,30127,"Tuesday, October 1, 2013",Standard Class,683.039978,683.039978,0,1,Technology,South


In [94]:
print("Missing values:")
print(df[
    [
        "Discount",
        "Total Sales Raw",
        "Profit",
        "Quantity",
        "Shipping Cost",
        "Market",
        "Region",
        "Ship Mode",
        "Category"
    ]
].isnull().sum())

print("\nNumber of unique values:")
print(df[
    [
        "Market",
        "Region",
        "Ship Mode",
        "Category"
    ]
].nunique())

Missing values:
Discount           0
Total Sales Raw    0
Profit             0
Quantity           0
Shipping Cost      0
Market             0
Region             0
Ship Mode          0
Category           0
dtype: int64

Number of unique values:
Market        7
Region       13
Ship Mode     4
Category      3
dtype: int64


In [95]:
def normality_test(data):
    data = pd.Series(data).dropna()
    statistic, p_value = stats.normaltest(data)

    if p_value < 0.05:
        result = "Not Normal"
    else:
        result = "Normal"

    return statistic, p_value, result


def check_groups(data, group_col, value_col):
    
    groups = {
        name: group[value_col].dropna()
        for name, group in data.groupby(group_col)
    }

    print(f"\n{group_col} vs {value_col}")

    # Normality
    for name, values in groups.items():
        _, p, result = normality_test(values)
        print(f"{name}: Normality p-value = {p:.6e} : {result}")

    # Levene
    _, levene_p = stats.levene(*groups.values())

    print(f"\nLevene test p-value: {levene_p:.6e}")

    return groups

In [96]:
groups_market = check_groups(
    df,
    "Market",
    "Profit"
)


Market vs Profit


APAC: Normality p-value = 0.000000e+00 : Not Normal
Africa: Normality p-value = 0.000000e+00 : Not Normal
Canada: Normality p-value = 8.140020e-104 : Not Normal
EMEA: Normality p-value = 0.000000e+00 : Not Normal
EU: Normality p-value = 0.000000e+00 : Not Normal
LATAM: Normality p-value = 0.000000e+00 : Not Normal
US: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 1.876851e-28


In [97]:
groups_region = check_groups(
    df,
    "Region",
    "Total Sales Raw"
)


Region vs Total Sales Raw
Africa: Normality p-value = 0.000000e+00 : Not Normal
Canada: Normality p-value = 4.289916e-120 : Not Normal
Caribbean: Normality p-value = 0.000000e+00 : Not Normal
Central: Normality p-value = 0.000000e+00 : Not Normal
Central Asia: Normality p-value = 0.000000e+00 : Not Normal
EMEA: Normality p-value = 0.000000e+00 : Not Normal
East: Normality p-value = 0.000000e+00 : Not Normal
North: Normality p-value = 0.000000e+00 : Not Normal
North Asia: Normality p-value = 0.000000e+00 : Not Normal
Oceania: Normality p-value = 0.000000e+00 : Not Normal
South: Normality p-value = 0.000000e+00 : Not Normal
Southeast Asia: Normality p-value = 0.000000e+00 : Not Normal
West: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 3.216410e-76


In [98]:
groups_ship = check_groups(
    df,
    "Ship Mode",
    "Shipping Cost"
)


Ship Mode vs Shipping Cost
First Class: Normality p-value = 0.000000e+00 : Not Normal
Same Day: Normality p-value = 0.000000e+00 : Not Normal
Second Class: Normality p-value = 0.000000e+00 : Not Normal
Standard Class: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 2.503390e-201


In [99]:
groups_category = check_groups(
    df,
    "Category",
    "Profit"
)


Category vs Profit


Furniture: Normality p-value = 0.000000e+00 : Not Normal
Office Supplies: Normality p-value = 0.000000e+00 : Not Normal
Technology: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 0.000000e+00


Market vs Profit

In [111]:
stat_market, p_market = stats.kruskal(*groups_market.values())

print("Market vs Profit")
print("Test: Kruskal-Wallis")
print(f"Statistic: {stat_market:.4f}")
print(f"P-value: {p_market:.6e}")

if p_market < 0.05:
    print("Reject H0")
    print("Result: Significant difference exists.")
else:
    print("Fail to Reject H0")
    print("Result: No significant difference found.")

Market vs Profit
Test: Kruskal-Wallis
Statistic: 421.5830
P-value: 6.385676e-88
Reject H0
Result: Significant difference exists.


**Profit varies significantly across markets. Management should monitor market-level profitability to identify underperforming markets.**

Region vs Total Sales

In [110]:
stat_region, p_region = stats.kruskal(*groups_region.values())

print("Region vs Total Sales")
print("Test: Kruskal-Wallis")
print(f"Statistic: {stat_region:.4f}")
print(f"P-value: {p_region:.6e}")

if p_region < 0.05:
    print("Reject H0")
    print("Result: Significant difference exists.")
else:
    print("Fail to Reject H0")
    print("Result: No significant difference found.")

Region vs Total Sales
Test: Kruskal-Wallis
Statistic: 3416.2117
P-value: 0.000000e+00
Reject H0
Result: Significant difference exists.


**Sales differ significantly across regions. Management should focus on low-performing regions and identify opportunities for growth.**

Ship Mode vs Shipping Cost

In [109]:
stat_ship, p_ship = stats.kruskal(*groups_ship.values())

print("Ship Mode vs Shipping Cost")
print("Test: Kruskal-Wallis")
print(f"Statistic: {stat_ship:.4f}")
print(f"P-value: {p_ship:.6e}")

if p_ship < 0.05:
    print("Reject H0")
    print("Result: Significant difference exists.")
else:
    print("Fail to Reject H0")
    print("Result: No significant difference found.")

Ship Mode vs Shipping Cost
Test: Kruskal-Wallis
Statistic: 1336.0334
P-value: 2.234667e-289
Reject H0
Result: Significant difference exists.


**Shipping costs differ significantly across shipping modes. Management should review delivery costs to identify opportunities for logistics optimization.**

Category vs Profit

In [108]:
stat_category, p_category = stats.kruskal(*groups_category.values())

print("Category vs Profit")
print("Test: Kruskal-Wallis")
print(f"Statistic: {stat_category:.4f}")
print(f"P-value: {p_category:.6e}")

if p_category < 0.05:
    print("Reject H0")
    print("Result: Significant difference exists.")
else:
    print("Fail to Reject H0")
    print("Result: No significant difference found.")

Category vs Profit
Test: Kruskal-Wallis
Statistic: 1752.7204
P-value: 0.000000e+00
Reject H0
Result: Significant difference exists.



**Profit differs significantly across product categories. Management should identify high- and low-profit categories to improve product strategy.**

Discount vs Total Sales

In [107]:
data_discount = df[
    ["Discount", "Total Sales Raw"]
].dropna()

print("Discount vs Total Sales")
print("N =", len(data_discount))

_, p_discount, normal_discount = normality_test(
    data_discount["Discount"]
)

_, p_sales, normal_sales = normality_test(
    data_discount["Total Sales Raw"]
)

print(f"Discount normality p-value: {p_discount:.6e} : {normal_discount}")
print(f"Sales normality p-value: {p_sales:.6e} : {normal_sales}")

if normal_discount == "Normal" and normal_sales == "Normal":
    statistic, p_value = stats.pearsonr(
        data_discount["Discount"],
        data_discount["Total Sales Raw"]
    )
    test_name = "Pearson Correlation"
else:
    statistic, p_value = stats.spearmanr(
        data_discount["Discount"],
        data_discount["Total Sales Raw"]
    )
    test_name = "Spearman Correlation"

print(f"\nSelected test: {test_name}")
print(f"Correlation: {statistic:.4f}")
print(f"P-value: {p_value:.6e}")

if p_value < 0.05:
    print("Reject H0")
    print("Result: Significant relationship exists.")
else:
    print("Fail to Reject H0")
    print("Result: No significant relationship found.")

Discount vs Total Sales
N = 49670
Discount normality p-value: 0.000000e+00 : Not Normal
Sales normality p-value: 0.000000e+00 : Not Normal

Selected test: Spearman Correlation
Correlation: -0.0718
P-value: 8.548575e-58
Reject H0
Result: Significant relationship exists.


**Discount has a significant but very weak negative relationship with Sales. Discount strategies should be evaluated carefully, as the relationship with Sales is minimal.**

Quantity vs Profit

In [106]:
data_quantity = df[
    ["Quantity", "Profit"]
].dropna()

print("Quantity vs Profit")
print("N =", len(data_quantity))

_, p_quantity, normal_quantity = normality_test(
    data_quantity["Quantity"]
)

_, p_profit, normal_profit = normality_test(
    data_quantity["Profit"]
)

print(f"Quantity normality p-value: {p_quantity:.6e} : {normal_quantity}")
print(f"Profit normality p-value: {p_profit:.6e} : {normal_profit}")

if normal_quantity == "Normal" and normal_profit == "Normal":
    statistic, p_value = stats.pearsonr(
        data_quantity["Quantity"],
        data_quantity["Profit"]
    )
    test_name = "Pearson Correlation"
else:
    statistic, p_value = stats.spearmanr(
        data_quantity["Quantity"],
        data_quantity["Profit"]
    )
    test_name = "Spearman Correlation"

print(f"\nSelected test: {test_name}")
print(f"Correlation: {statistic:.4f}")
print(f"P-value: {p_value:.6e}")

if p_value < 0.05:
    print("Reject H0")
    print("Result: Significant relationship exists.")
else:
    print("Fail to Reject H0")
    print("Result: No significant relationship found.")

Quantity vs Profit
N = 49670
Quantity normality p-value: 0.000000e+00 : Not Normal
Profit normality p-value: 0.000000e+00 : Not Normal

Selected test: Spearman Correlation
Correlation: 0.1986
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.


**Quantity has a significant but weak positive relationship with Profit. Increasing quantity alone may not substantially improve profitability.**